# Distances Between Probability Measures [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xavimaass/computational_optimal_transport_2026/blob/main/assignments/HW3.ipynb)

**Student Name:** [Your Name Here]

In this notebook, we will explore different ways to measure distances between probability distributions.

In [ ]:
# If running on Google Colab, you might need to install POT (Python Optimal Transport library).
# !pip install POT

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm
import ot

# Plotting settings
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['axes.grid'] = True

## Warm-Up: Wasserstein distance in 1D

In the lectures, you have seen that the 1D wasserstein distance has a closed-form solution that can be expressed via the quantile functions:
$$W_p(\mu, \nu) = \Big(\int_0^1 |Q_\mu(t) - Q_\nu(t)|^p dt \Big)^{1/p}$$
where $Q_\mu$, $Q_\nu$ are the quantile (inverse CDF) functions.

**Exercise 1.** Implement the 1D wasserstein-p distance between 2 point clouds (of potentially different sizes $n$ and $m$)

*Hint: Because the empirical distributions are discrete, their quantile functions $Q(t)$ are step functions. If $n = m$, you could simply sort both arrays and compute the distance element-wise. However, when $n \neq m$, the "steps" of the quantile functions don't align. To compute the exact integral $\int_0^1 |Q_\mu(t) - Q_\nu(t)|^p dt$, you need to evaluate the area of the piecewise constant segments.*

**Suggested Steps:**
- Sort both arrays.
- Define the CDF levels. Create arrays representing the probability levels for each distribution: $1/n, 2/n, \dots, 1$ and $1/m, 2/m, \dots, 1$ (e.g. use `np.arange`).
- Merge the CDF levels by using `np.concatenate` + `np.unique`. These represent the boundaries on the $[0, 1]$ interval where *either* quantile function might jump.
- Evaluate both quantile functions at these levels (Tip: `np.searchsorted` is highly efficient for finding which quantile step a probability level falls into; e.g. `u_quantiles = u_sorted[np.searchsorted(u_cdf, all_levels, side='left')]`).
- Compute the widths of each probability interval (Tip: you can use `np.diff` on an array `np.concatenate([0.0], all_levels)`).
- Compute the integral as the sum of `width * |u_quantiles - v_quantiles|^p` (i.e. a weighted sum).
- Return the \(p\)-th root (unless `return_pth_power=True`).

In [ ]:
def wasserstein_p_1d(u: np.ndarray, v: np.ndarray, p: float = 2, return_pth_power: bool = False) -> float:
    """
    Compute the 1D Wasserstein-p distance between two point clouds (of possibly different size).

    Args:
        u: 1D array of samples from the first distribution; of shape (n,).
        v: 1D array of samples from the second distribution; of shape (m,).
        p: Order of the distance (p >= 1). Defaults to 2.
        return_pth_power: If True, return the p-th power of the distance
            instead of the distance itself. Defaults to False.

    Returns:
        The Wasserstein-p distance as a float (or its pth power).
    """
    if p < 1:
        raise ValueError(f"p must be >= 1, got {p}")

    u = np.asarray(u, dtype=float).ravel()
    v = np.asarray(v, dtype=float).ravel()

    n, m = len(u), len(v)
    if n == 0 or m == 0:
        raise ValueError("Input arrays must be non-empty")

    # YOUR CODE HERE
    # SORT THE ARRAYS
    # FIND THE CDF LEVELS
    # FIND THE QUANTILES
    # COMPUTE THE INTEGRAL USING THE QUANTILES AND THE CDF LEVELS

    integral = None
    return float(integral) if return_pth_power else float(integral ** (1.0 / p))

In 1D, for two Gaussians $\mu_1 = \mathcal{N}(\mu_1, \sigma_1^2)$ and $\mu_2 = \mathcal{N}(\mu_2, \sigma_2^2)$, we exactly have $W_2^2(\mu_1,\mu_2) = (\mu_1 - \mu_2)^2 + (\sigma_1 - \sigma_2)^2$.

**Exercise 2.** Generate samples from two 1D Gaussians $\mathcal{N}(\mu_1, \sigma_1^2)$ ($n=10000$) and $\mathcal{N}(\mu_2, \sigma_2^2)$ ($m=20000$) (you may choose $\mu_i$, $\sigma_i$ as you wish!) And use this to verify that your implementation of `wasserstein_p_1d` is correct by comparing its output with the formula provided above (bear in mind that your implementation is only an approximation of the real distance!).

In [ ]:
def sanity_check_wasserstein_1d(mu1=0.0, sigma1=1.0, mu2=5.0, sigma2=1.0, n=10000, m=20000):
    # YOUR CODE HERE
    # Generate the samples!
    w2_analytic = None
    w2_empirical = None

    # Plot densities
    t = np.linspace(-4, 10, 500)
    plt.plot(t, norm.pdf(t, mu1, sigma1), label=f'N({mu1},{sigma1})')
    plt.plot(t, norm.pdf(t, mu2, sigma2), label=f'N({mu2},{sigma2})')
    plt.title(f"Distance between Gaussians (real $W_2^2$ ≈ {w2_analytic:.1f}, implemented $W_2^2$ ≈ {w2_empirical:.1f})")
    plt.legend()
    plt.show()


In [ ]:
n, m = 10000, 20000
# example values! You may change these as you like.
mu1, sigma1 = 0.0, 1.0
mu2, sigma2 = 5.0, 1.0

sanity_check_wasserstein_1d(mu1, sigma1, mu2, sigma2, n, m)

##

## Distances between probability measures in higher dimensions

In [ ]:
## UTILITY FUNCTION
def _ensure_arrays(x,y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    if x.ndim == 1:
        x = x[:, None]
    if y.ndim == 1:
        y = y[:, None]
    return x, y


### Wasserstein Distance

The **Wasserstein distance** measures the optimal transport cost for moving mass from one distribution to the other (using $\mathrm{dist}^p$). Formally, for two distributions $\mu$ and $\nu$ on $\mathbb{R}^d$, the $p$-Wasserstein distance is:

$$W_p(\mu, \nu) := \left( \inf_{\gamma \in \Pi(\mu,\nu)} \int \|x - y\|^p \, d\gamma(x, y) \right)^{1/p}$$

where $\Pi(\mu, \nu)$ is the set of couplings with marginals $\mu$ and $\nu$.

Given empirical samples, this becomes a finite linear program, that we can solve using the dedicated python library `POT` (see [here](https://pythonot.github.io/auto_examples/plot_quickstart_guide.html)). The usual way of doing this is:
- Build the cost matrix $C_{ij} = \|x_i - y_j\|^p$ by using `C = ot.dist(x, y, metric='euclidean') ** p`.
- Solve the OT problem `float(ot.emd2(a, b, C))`, where `a` and `b`are arrays of size $n$ and $m$ respectively, which contain uniform weights for each sample (i.e. $a_i = 1/n$, $b_j = 1/m$).
- Take the $p$-th root (if `return_pth_power=False`).

**Exercise 3.** Complete the function below to implement the Wasserstein $p$ distance between 2 point clouds.

In [ ]:
def wasserstein_p_empirical(x, y, p=2, return_pth_power=False):
    """Compute the empirical Wasserstein-p distance between two point clouds (of possibly different size) using POT.
    x: array of shape (n, d) - samples from the first distribution
    y: array of shape (m, d) - samples from the second distribution
    p: Order of the distance (p >= 1). Defaults to 2.
    return_pth_power: If True, return the p-th power of the distance
        instead of the distance itself. Defaults to False.
    """
    if p < 1:
        raise ValueError(f"p must be >= 1, got {p}")

    x, y = _ensure_arrays(x,y)

    # YOUR CODE HERE

    wp_p = None
    return wp_p if return_pth_power else wp_p ** (1.0 / p)

### Sliced Wasserstein (SW) distance
The **Sliced Wasserstein (SW)** distance is an attempt at circumventing the curse of dimensionality of the Wasserstein distance (we will see this in the next section) by reducing the problem to a collection of cheap one-dimensional problems. For $\theta \in \mathbb{S}^{d-1}$, let $\Pi^\theta(x) = \langle \theta, x \rangle$ be the scalar projection of $x$ onto direction $\theta$. The pushed-forward measure $\Pi^\theta_\# \mu$ is simply the distribution of those projected scalars. The sliced Wasserstein distance then averages the 1D Wasserstein cost over all directions:

$$\text{SW}_p(\mu, \nu) := \left( \int_{\mathbb{S}^{d-1}} W_p^p\!\left(\Pi^\theta_\#\mu,\, \Pi^\theta_\#\nu\right) \sigma(d\theta) \right)^{1/p}$$

where $\sigma$ is the uniform measure on $\mathbb{S}^{d-1}$.

For implementing this, the integral over the sphere is approximated by averaging over $L$ random directions, each drawn uniformly from $\mathbb{S}^{d-1}$. In 1D, $W_p^p$ reduces to a simple sort-and-match, making each projection $O(n \log n)$ — far cheaper than solving the full OT problem in $d$ dimensions.

**Exercise 4.** Complete the function below to implement the Sliced Wasserstein $p$ distance between 2 point clouds. You may use the `wasserstein_p_1d` distance implemented in a previous exercise.

In [ ]:
def sliced_wasserstein_p(x, y, n_projections=50, p=2, return_pth_power=False):
    """Compute the empirical Sliced-Wasserstein-p between two point clouds (of possibly different size).
    x: array of shape (n, d) - samples from the first distribution
    y: array of shape (m, d) - samples from the second distribution
    p: Order of the distance (p >= 1). Defaults to 2.
    return_pth_power: If True, return the p-th power of the distance
        instead of the distance itself. Defaults to False.
    """
    if p < 1:
        raise ValueError(f"p must be >= 1, got {p}")

    x,y = _ensure_arrays(x,y)

    # YOUR CODE HERE

    swp_p = None
    return swp_p if return_pth_power else swp_p ** (1.0 / p)

### The Maximum Mean Discrepancy (MMD)
The **Maximum Mean Discrepancy (MMD)** is a kernel-based quantity that measures the "closeness" between probability distributions (but it **doesn't necessarily define a distance!**). Given two distributions $\mu$ and $\nu$ on $\mathbb{R}^d$, the MMD is defined as the largest difference in expected values of a function $f$ drawn from the unit ball of a reproducing kernel Hilbert space (RKHS) $\mathcal{H}$:

$$\text{MMD}(\mu, \nu) = \sup_{\substack{f \in \mathcal{H} \\ \|f\|_\mathcal{H} \le 1}} \left| \int f \, d\mu - \int f \, d\nu \right|$$

Expressing elements of the RKHS via the corresponding kernel, this admits the equivalent representation:

$$\text{MMD}^2(\mu, \nu) = \iint k(x,y)\,\mu(dx)\mu(dy) + \iint k(x,y)\,\nu(dx)\nu(dy) - 2\iint k(x,y)\,\mu(dx)\nu(dy)$$

Given i.i.d. samples $X_1, \ldots, X_m \sim \mu$ and $Y_1, \ldots, Y_n \sim \nu$, we can estimate this quantity via the empirical estimator, in which we simply replace integrals with sample averages over the kernel matrix (i.e. `kernel(x, x).mean()`, `kernel(y, y).mean()` and `kernel(x, y).mean()`).

**Exercise 5.** Complete the function below to implement the MMD between two point clouds. We have provided some kernel functions commonly used in practice.

In [ ]:
## IMPLEMENTING SOME KERNELS FOR MMD
def rbf(a, b, sigma=1.0):
    return np.exp(-ot.dist(a, b, metric='sqeuclidean') / (2 * sigma**2))

def laplacian(a, b, sigma=1.0):
    return np.exp(-ot.dist(a, b, metric='euclidean') / sigma)

def linear(a, b):
    return a @ b.T

def poly(a, b, degree=3, c=1.0):
    return (a @ b.T + c) ** degree

In [ ]:
def mmd(x, y, kernel=rbf, return_squared=False):
    """Compute the empirical MMD (according to a pluggable kernel) between two point clouds.
    x: array of shape (n, d) - samples from the first distribution
    y: array of shape (m, d) - samples from the second distribution
    kernel: The kernel function to use. Defaults to the RBF kernel.
    return_squared: If True, return the square of the MMD instead of the MMD itself. Defaults to False.
    """
    x, y = _ensure_arrays(x,y)

    # YOUR CODE HERE
    mmd2 = None
    return mmd2 if return_squared else np.sqrt(mmd2)

Just for completeness, we provide an approximate implementation of the Total Variation Distance.

In [ ]:
from scipy.stats import gaussian_kde

def tv_distance(x, y, n_points=10000):
    """TV distance approximation via KDE (to estimate the densities)."""
    x, y = _ensure_arrays(x,y)
    if x.shape[1] != y.shape[1]:
        raise ValueError("x and y must have the same ambient dimension")

    kde_x = gaussian_kde(x.T, 0.01)
    kde_y = gaussian_kde(y.T, 0.01)

    pooled = np.concatenate([x, y], axis=0)
    idx = np.random.choice(pooled.shape[0], size=min(n_points, len(pooled)), replace=True)
    points = pooled[idx].T

    p = kde_x(points)
    q = kde_y(points)

    w = 0.5 * (p + q)
    return np.mean(np.abs(p - q) / w)

### Putting it all together

**Exercise 6.** Complete the function `sample_clouds` below, which generates two point clouds in $\mathbb{R}^2$ with `n_samples` points:

1. $x$: a mixture of 3 Gaussians centered at the vertices of an equilateral triangle inscribed in a circle of radius `vertex_radius`, each with noise standard deviation `noise_scale` (this parameter won't be playing an important role). In other words, the means of these gaussians are placed at $(r\cos(\theta), r\sin(\theta))$, where $r$ = `vertex_radius` and $\theta$ is one of $\{0, 2 \pi/ 3, 4\pi / 3\}$.
2. $y$: a single isotropic Gaussian centered at the origin with standard deviation `gaussian_std`.

As a sanity check, use the provided plotting function to visualize the obtained point clouds. For instance, check that when taking `vertex_radius = 0` and `gaussian_std = noise_scale`, the two point clouds _look indistinguishable_.


In [ ]:
def sample_clouds(vertex_radius, gaussian_std, n_samples, noise_scale=0.25, seed=0):
    """Generate a pair of point clouds: triangle-vertex Gaussian mixture and centered Gaussian.

    Args:
        vertex_radius: Distance from origin to each triangle vertex.
        gaussian_std: Standard deviation of the centered Gaussian.
        n_samples: Number of samples to generate for each distribution.
        noise_scale: Standard deviation of noise added to triangle vertices.
        seed: Random seed for reproducibility.

    Returns:
        x: Point cloud as 3-component Gaussian mixture (n_samples, 2).
        y: Point cloud as centered Gaussian (n_samples, 2).
    """
    # YOUR CODE HERE
    x = None # 3-component gaussian mixture
    y = None # centered gaussian
    return x, y

In [ ]:
def plot_2d_clouds(x2d, y2d):
    # Visualize both point clouds
    plt.figure(figsize=(8, 8))
    plt.scatter(x2d[:, 0], x2d[:, 1], s=10, alpha=0.45, label='Gaussian mixture samples')
    plt.scatter(y2d[:, 0], y2d[:, 1], s=10, alpha=0.45, label='Gaussian samples (mean=0)')
    plt.gca().set_aspect('equal', 'box')
    plt.title('2D point clouds: triangle-vertex Gaussian mixture vs centered Gaussian')
    plt.xlim(-2.5, 2.5)
    plt.ylim(-2.5, 2.5)
    plt.legend()
    plt.show()

In [ ]:
# Generate point clouds
x2d, y2d = sample_clouds(
    vertex_radius=1.5,
    gaussian_std=0.75,
    n_samples=500
)

plot_2d_clouds(x2d, y2d)

In [ ]:
# Generate point clouds
x2d, y2d = sample_clouds(
    vertex_radius=0,
    gaussian_std=0.25,
    n_samples=500
)

plot_2d_clouds(x2d, y2d)

**Exercise 7.** Complete the function `compute_metric_grids` below, which evaluates several distributional metrics over a 2D parameter grid. For each pair `gaussian_std_i`, `vertex_radius_j` in the grid, it generates two point clouds using the function `sample_clouds` and evaluates each distance metric in `metrics`, storing results in a dictionary of 2D arrays of shape `len(std_grid)`, `len(radius_grid)`.

In [ ]:
def compute_metric_grids(metrics, std_grid, radius_grid, n_heatmap = 180):
    """ Compute the metric values over a grid of (gaussian_std, vertex_radius) pairs.
    metrics: dict of metric_name -> metric_function(x,y) that computes a distance between two point clouds.
    std_grid: 1D array of gaussian_std values to evaluate.
    radius_grid: 1D array of vertex_radius values to evaluate.
    """

    grids = {name: np.full((len(std_grid), len(radius_grid)), np.nan, dtype=float) for (name, _) in metrics.items()}
    for i, gaussian_std_i in enumerate(std_grid):
        for j, vertex_radius_j in enumerate(radius_grid):
            # YOUR CODE HERE
            # GENERATE THE DATA!
            for name, metric in metrics.items():
                try:
                    grids[name][i, j] = None # COMPUTE THE METRIC !
                except Exception:
                    grids[name][i, j] = np.nan
    return grids

**Exercise 8.** Run `compute_metric_grids` on a given metric_catalog containing metrics such as:
$W_1$; $W_2$; $SW_1$; $SW_2$; $MMD$ with RBF, Laplacian, Linear and Polynomial kernels; and the (approximate) Total Variation distance.

Use the plotting function to visualize the obtained grid. Comment on the obtained plot: Do all metrics behave similarly? What qualitative differences do you observe?

In [ ]:
# Flexible heatmap panel over (gaussian_std, vertex_radius)
grid_size = 16
param_min, param_max = 0.0, 1.0
std_vals = np.linspace(param_min, param_max, grid_size)
radius_vals = np.linspace(param_min, param_max, grid_size)

metric_catalog = {
    "W1": lambda x, y: wasserstein_p_empirical(x, y, p=1),
    "W2": None, # YOUR CODE HERE
    "SW1": None, # YOUR CODE HERE
    "SW2": None, # YOUR CODE HERE
    "MMD-RBF": None, # YOUR CODE HERE
    "MMD-Laplacian": None, # YOUR CODE HERE
    "MMD-Linear": None, # YOUR CODE HERE
    "MMD-Polynomial": None, # YOUR CODE HERE
    "TV (KDE-MC approx)": tv_distance,
}

In [ ]:
def plot_metric_panel(grids, std_grid, radius_grid, mode='imshow', cmap_name='viridis'):
    n_metrics = len(grids)
    ncols = 3
    nrows = int(np.ceil(n_metrics / ncols))
    fig, axes = plt.subplots(
        nrows, ncols, figsize=(5.8 * ncols, 4.7 * nrows), constrained_layout=True
    )
    axes = np.atleast_1d(axes).ravel()

    extent = [radius_grid.min(), radius_grid.max(), std_grid.min(), std_grid.max()]
    X, Y = np.meshgrid(radius_grid, std_grid)

    for ax, (metric_name, grid_values) in zip(axes, grids.items()):
        if mode == 'contourf':
            im = ax.contourf(X, Y, grid_values, levels=20, cmap=cmap_name)
        else:
            im = ax.imshow(
                grid_values,
                origin='lower',
                extent=extent,
                aspect='auto',
                cmap=cmap_name,
            )
        ax.set_title(metric_name)
        ax.set_xlabel('vertex_radius')
        ax.set_ylabel('gaussian_std')
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

    for ax in axes[n_metrics:]:
        ax.axis('off')

    plt.suptitle('Distance heatmaps over (gaussian_std, vertex_radius)', y=1.02)
    plt.show()

In [ ]:
metric_grids = compute_metric_grids(
    metrics=metric_catalog,
    std_grid=std_vals,
    radius_grid=radius_vals,
 )
plot_metric_panel(metric_grids, std_vals, radius_vals)

[YOUR ANSWER HERE]

## The Curse of Dimensionality

The empirical Wasserstein distance converges to the true continuous Wasserstein distance at a rate of roughly $n^{-1/d}$, where $n$ is the number of samples and $d$ is the dimension. This represents the _curse of dimensionality_, because for large dimensions, the convergence will be very slow.

To illustrate this, we will focus on the example of computing the distance between $\mu = \mathcal{N}(0, I_d)$ and **itself** (e.g. $W_p(\mu, \mu)$). If we had access to the full measure $\mu$, the distance should be 0. However, since we have finitely many samples to _approximate this value_, we will observe how fast our _empirical_ metrics approach $0$ as the number of samples increases.

In [ ]:
# PLOTTING FUNCTIONS
def plot_dimension_convergence(sample_sizes, results_by_dim, dimensions, ylabel, title, label_prefix='', marker='o', linestyle='-'):
    """Plot mean convergence curves with trial-based error bars on a log-log scale."""
    plt.figure()
    for d in dimensions:
        trial_matrix = np.asarray(results_by_dim[d], dtype=float)
        means = trial_matrix.mean(axis=1)
        stds = trial_matrix.std(axis=1, ddof=1) if trial_matrix.shape[1] > 1 else np.zeros_like(means)

        # Keep lower error bars positive for log-scale plotting.
        yerr = np.minimum(stds, 0.95 * means)
        label = f'{label_prefix}d={d}' if label_prefix else f'd={d}'
        line = plt.loglog(sample_sizes, means, marker=marker, linestyle=linestyle, label=label)[0]
        plt.errorbar(
            sample_sizes,
            means,
            yerr=yerr,
            fmt='none',
            ecolor=line.get_color(),
            elinewidth=1.2,
            capsize=3,
            alpha=0.8,
        )

    plt.xlabel('Number of samples (n)')
    plt.xlim(left=max(sample_sizes[0]-10, 1))
    plt.ylabel(ylabel)
    plt.title(title)
    plt.legend()
    plt.show()

In [ ]:
# PLOTTING FUNCTIONS
def plot_scaling_comparison(n_vals, d_max, series, title=None):
    """Plot empirical scaling curves and their fixed-rate fits."""
    plt.figure(figsize=(10, 6))

    for spec in series.values():
        data_label = spec['data_label'].replace('{d}', str(d_max))
        fit_label = spec['fit_label'].replace('{d}', str(d_max))

        line = plt.loglog(
            n_vals,
            spec['values'],
            f"{spec['marker']}-",
            label=data_label,
        )[0]
        plt.loglog(
            n_vals,
            spec['fit'],
            '--',
            color=line.get_color(),
            alpha=0.2,
            label=fit_label,
        )

    plt.xlabel('Number of samples (n)')
    plt.ylabel('Distance (log scale)')
    plt.title(title or f'Scaling with n at max dimension d={d_max}')
    plt.legend()
    plt.show()

def fit_power_law_with_fixed_exponent(n, y, alpha):
    """Fit y ~ C * n^{-alpha} by estimating C in log-space."""
    log_c = np.mean(np.log(y) + alpha * np.log(n))
    c = np.exp(log_c)
    return c * n ** (-alpha)


def plot_scaling_from_results(results, rates, d_max):
    """Build and plot scaling comparison from experiment outputs."""
    n_vals = np.asarray(results['sample_sizes'], dtype=float)

    labels = {
        'W2': {
            'marker': 'o',
            'data_label': 'Empirical W2 (d={d})',
            'fit_label': r'Fit $\propto n^{-1/{d}}$',
        },
        'SW2': {
            'marker': 's',
            'data_label': 'Empirical SW2 (d={d})',
            'fit_label': r'Fit $\propto n^{-1/2}$ (SW2)',
        },
        'MMD': {
            'marker': '^',
            'data_label': 'Empirical MMD (d={d})',
            'fit_label': r'Fit $\propto n^{-1/2}$ (MMD)',
        },
    }

    series = {}
    for metric_name, style in labels.items():
        metric_mean = np.asarray(results[metric_name][d_max], dtype=float).mean(axis=1)
        series[metric_name] = {
            'values': metric_mean,
            'fit': fit_power_law_with_fixed_exponent(n_vals, metric_mean, rates[metric_name]),
            **style,
        }
    plot_scaling_comparison(n_vals, d_max, series)

**Exercise 9.** Complete the function `run_dimension_experiment`, to do the following:
- Iterate over `dimensions` and `sample_sizes` and also over `range(trials)`.
- At each trial for a given value of `d`, `n`, sample `x`and `y`as standard gaussians $\mathcal{N}(0,1)$; and compute the `metric_fn(x,y)`.
- You should save the results in a dictionary: `results_by_dim` that maps each dimension `d` to an array of shape
    `(len(sample_sizes), trials)`.


In [ ]:
def run_dimension_experiment(dimensions, sample_sizes, trials, metric_fn):
    """Run a dimension-varying experiment and return per-trial metric values by dimension.
    dimensions: List of dimensions to test.
    sample_sizes: List of sample sizes to test.
    trials: Number of independent trials to run for each (dimension, sample_size) pair.
    metric_fn: Function that takes two point clouds and returns the distance metric.

    Returns a dict mapping each dimension d to an array of shape (len(sample_sizes), trials)
        containing the distance metric values for each trial and sample size.
    """
    results_by_dim = {
        d: np.empty((len(sample_sizes), trials), dtype=float) for d in dimensions
    }

    # YOUR CODE HERE

    return results_by_dim

In [ ]:
## Let's define the parameters for all the runs!
dimensions = [1, 2, 5, 10, 50]
sample_sizes = [50, 100, 200, 500, 1000, 2000]
trials = 5

**Exercise 10. a)** Run the experiment for the Wasserstein distance. Comment on what you observe.

In [ ]:
results_w2 = run_dimension_experiment(
    dimensions=dimensions,
    sample_sizes=sample_sizes,
    trials=trials,
    metric_fn=wasserstein_p_empirical
)

plot_dimension_convergence(
    sample_sizes=sample_sizes,
    results_by_dim=results_w2,
    dimensions=dimensions,
    ylabel='Empirical W2 (True = 0)',
    title='Wasserstein',
)

[YOUR ANSWER HERE]

**Exercise 10. b)** Run the experiment for the Sliced Wasserstein distance. Comment on what you observe.

In [ ]:
# --- SOLUTION ---
results_sw = run_dimension_experiment(
    dimensions=dimensions,
    sample_sizes=sample_sizes,
    trials=trials,
    metric_fn=lambda x, y: sliced_wasserstein_p(x, y, n_projections=50)
)

plot_dimension_convergence(
    sample_sizes=sample_sizes,
    results_by_dim=results_sw,
    dimensions=dimensions,
    ylabel='Empirical Sliced-W2 (True = 0)',
    title='Sliced-Wasserstein',
    label_prefix='SW, ',
)

[YOUR ANSWER HERE]

**Exercise 10. c)** Run the experiment for the MMD. Comment on what you observe.

In [ ]:
sigma=1.0
results_mmd = run_dimension_experiment(
    dimensions=dimensions,
    sample_sizes=sample_sizes,
    trials=trials,
    metric_fn=lambda x, y: mmd(x, y, kernel=lambda a, b: rbf(a, b, sigma=sigma))
)

plot_dimension_convergence(
    sample_sizes=sample_sizes,
    results_by_dim=results_mmd,
    dimensions=dimensions,
    ylabel='Empirical MMD (True = 0)',
    title='MMD',
    label_prefix='MMD, ',
)

[YOUR ANSWER HERE]

**Exercise 11.** Run the following cell to observe the scaling of the distance values for a given value of `d`. Comment on the observed scalings. Did these _alternative_ measures (SW and MMD) allow us to _escape_ the curse of dimensionality?

In [ ]:
scaling_results = {
    'sample_sizes': sample_sizes,
    'W2': results_w2,
    'SW2': results_sw,
    'MMD': results_mmd,
}
d_val = max(dimensions)

rates = {
    'W2': 1.0 / d_val,
    'SW2': 0.5,
    'MMD': 0.5,
}

plot_scaling_from_results(scaling_results, rates, d_max=d_val)

[YOUR ANSWER HERE]